In [44]:
import pandas as pd

df_raw = pd.read_excel("GSAF5.xls", engine="xlrd")
print(df_raw.shape)

(7125, 23)


In [45]:
ACTIVITY_CORRECTIONS = {
    "swimmingq": "swimming",
    "spearfishiong": "spearfishing",
    "snorkelling": "snorkeling",
    "boggie boarding": "boogie boarding",
}
LOCATION_CORRECTIONS = {
    "300 miles east of Easter Island": "300 miles East of Easter Island",
    "Cheynes beach": "Cheynes Beach",
    "Kei River mouth": "Kei River Mouth",
    "Lauderdale-By-The-Sea, Broward County": "Lauderdale-by-the-Sea, Broward County",
    "Ponce De Leon Inlet, Volusia County": "Ponce de Leon Inlet, Volusia County",
    "South coast": "South Coast",
}
MISSING = {"", "nan", "n/a", "na", "null", "none", "unknown"}
FATAL_VALID = ["Y", "N"]
INJURY_EVIDENCE = r"\b(?:fatal|died|death|drown\w*|remains|body washed|scaveng\w*)\b"

REASON_MISSING = "Ausente o desconocido"
REASON_INVALID = "Valor no valido; verificar fuente primaria"
REASON_EVIDENCE = "; muerte/restos mencionados: verificar causa y participacion del tiburon"
REASON_CONFLICT = "Posible contradiccion N / FATAL; conservar N hasta verificar fuente"
ACTIVITY_FILL = "unknown"

def normalize(s):
    """NFKC + colapsa espacios + strip, sobre una Serie. Los nulos se mantienen como NA."""
    return (
        s.astype("string")
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [46]:
def clean_activity(df):
    df = df.copy()
    activity = normalize(df["Activity"]).str.casefold()
    df["Activity"] = (
        activity
        .mask(activity.isin(MISSING))      # "", "nan", "unknown"... -> NA
        .replace(ACTIVITY_CORRECTIONS)     # typos conocidos
        .fillna(ACTIVITY_FILL)             # los NA -> "unknown"
    )
    return df


def clean_fatal(df):
    """Normaliza 'Fatal Y/N' a Y / N / UNKNOWN y anota en
    'Fatal_review_reason' los casos que hay que verificar a mano.
    Nunca deduce N a partir de lesiones ni Y a partir de restos humanos."""
    df = df.copy()

    fatal = normalize(df["Fatal Y/N"]).str.upper()
    injury = normalize(df["Injury"])

    clean = fatal.where(fatal.isin(FATAL_VALID), "UNKNOWN")
    unknown = clean.eq("UNKNOWN")
    missing = fatal.isna() | fatal.str.casefold().isin(MISSING)
    evidence = injury.str.contains(INJURY_EVIDENCE, case=False, na=False)
    conflict = clean.eq("N") & injury.str.contains(r"^\s*FATAL\b", case=False, na=False)

    reason = pd.Series("", index=df.index, dtype="object")
    reason[unknown & missing] = REASON_MISSING
    reason[unknown & ~missing] = REASON_INVALID
    reason[unknown & evidence] += REASON_EVIDENCE
    reason[conflict] = REASON_CONFLICT

    df["Fatal Y/N"] = clean
    df["Fatal_review_reason"] = reason
    return df

In [47]:
def report_cleaning(antes, despues, nombre, fill_label=None):
    nulos_antes, nulos_despues = antes.isna().sum(), despues.isna().sum()

    print(f"{nombre} - ANTES")
    print("  Filas:", len(antes))
    print("  Nulos:", nulos_antes)
    print("  Valores únicos:", antes.nunique())

    print(f"\n{nombre} - DESPUÉS")
    print("  Filas:", len(despues))
    print("  Nulos:", nulos_despues, f"({nulos_despues - nulos_antes:+d} vs antes)")
    if fill_label is not None:
        print(f"  '{fill_label}':", (despues == fill_label).sum())
    print("  Valores únicos:", despues.nunique(), f"({antes.nunique() - despues.nunique()} menos)")

    centinela = "<<NA>>"
    cambiadas = antes.fillna(centinela) != despues.fillna(centinela)
    cambios_reales = cambiadas & antes.notna()
    print("\nFilas modificadas:", cambiadas.sum(), f"(de ellas, sobre texto real: {cambios_reales.sum()})")

    comparacion = pd.DataFrame({"antes": antes, "despues": despues})

    print("\nHEAD (primeras 15 filas):")
    print(comparacion.head(15).to_string())

    print("\nMUESTRA ALEATORIA (15 filas):")
    print(comparacion.sample(min(15, len(comparacion)), random_state=42).to_string())

    print("\nEjemplos de cambios (excluyendo nulos):")
    print(comparacion[cambios_reales].head(15).to_string())

    print("\nTop 20 valores ANTES:")
    print(antes.value_counts(dropna=False).head(20).to_string())

    print("\nTop 20 valores DESPUÉS:")
    print(despues.value_counts(dropna=False).head(20).to_string())

    print("\nValores más raros ANTES (cola):")
    print(antes.value_counts().tail(20).to_string())

In [48]:
df = df_raw.copy()   # siempre parte del original

# --- Activity ---
antes = df["Activity"].astype("string")
df = clean_activity(df)
report_cleaning(antes, df["Activity"], "Activity")

# --- Fatal Y/N ---
print("\n" + "=" * 60 + "\n")
antes = df["Fatal Y/N"].astype("string")
df = clean_fatal(df)
report_cleaning(antes, df["Fatal Y/N"], "Fatal Y/N", fill_label="UNKNOWN")

print("\nMotivos de revisión Fatal:")
print(df["Fatal_review_reason"].replace("", pd.NA).value_counts().to_string())

Activity - ANTES
  Filas: 7125
  Nulos: 583
  Valores únicos: 1617

Activity - DESPUÉS
  Filas: 7125
  Nulos: 0 (-583 vs antes)
  Valores únicos: 1550 (67 menos)

Filas modificadas: 7097 (de ellas, sobre texto real: 6514)

HEAD (primeras 15 filas):
                          antes                      despues
0                      Swimming                     swimming
1                        Diving                       diving
2                      Swimming                     swimming
3                       Surfing                      surfing
4                       Surfing                      surfing
5                       Surfing                      surfing
6                  Spearfishing                 spearfishing
7                      Swimming                     swimming
8                       Surfing                      surfing
9           Wading on a sandbar          wading on a sandbar
10                Spearfishiong                 spearfishing
11                 

In [ ]:
# --- Resumen ---
print("\nFatal Y/N:")
print(df["Fatal Y/N"].value_counts().to_string())
print("\nActivity nulos:", df["Activity"].isna().sum())
print("Activity únicos:", df["Activity"].nunique())



Fatal Y/N:
Fatal Y/N
N          4986
Y          1496
UNKNOWN     643

Activity nulos: 0
Activity únicos: 1550

Casos a revisar a mano: 644
Fatal_review_reason
Ausente o desconocido                                                                                                 413
Ausente o desconocido; muerte/restos mencionados: verificar causa y participacion del tiburon                         219
Valor no valido; verificar fuente primaria; muerte/restos mencionados: verificar causa y participacion del tiburon      6
Valor no valido; verificar fuente primaria                                                                              5
Posible contradiccion N / FATAL; conservar N hasta verificar fuente                                                     1
